# Two-Tower Model — complementary products

**The goal of this notebook is to build a two-tower neural network (TTN) that
finds complementary products** — given an item a user is looking at, retrieve
the items that are bought *alongside* it rather than the items most similar to
it. A phone case complements a phone; another phone does not.

The approach follows **[Suggest, complement, inspire: story of Two Tower
recommendations at Allegro.com](https://arxiv.org/html/2508.03702v1)**
(Osowska-Kurczab, Nazarko, Marzec, Wojciechowska & Kremeňová, RecSys '25),
whose Complementary-TT model is the architecture this work is based on.

## Both towers describe items

This is the part that differs from the classic user/item two-tower setup, and
it shapes every column decision below: **the query tower and the candidate
tower both consume item information.** Neither tower is a user tower.

```
   query ITEM features                    candidate ITEM features
        │                                          │
   ┌────▼────┐                                ┌────▼────┐
   │  QUERY  │  product encoder               │CANDIDATE│  product encoder
   │  TOWER  │  (+ target category)           │  TOWER  │
   └────┬────┘                                └────┬────┘
        │                                          │
   q ∈ ℝ^d  ──────────  score = q · c  ──────────  c ∈ ℝ^d
```

Both towers share the same *architecture* — the paper's "Product Encoder":
each item attribute goes through its own embedding table, the vectors are
concatenated, passed through an MLP and L2-normalised. In the paper the query
tower is the only one modified for the complementary task: the query product
embedding is concatenated with a **target category embedding** drawn from a
one-to-many complementary-category mapping, while the candidate tower stays a
plain product encoder.

That mapping is what `complementary_cats_pairs/` produces —
`data/complementary_categories.pkl`, source category path → target category path,
scored by support and lift. §1 loads it. The co-purchase pairs that supply the
training positives come from the same package's `pairs.ipynb`.

Because both sides are items, per-user history is not a tower input at all and
this notebook does not load it. A user's history still shapes the data — it is
what defines which items count as co-purchased — but that work happens upstream,
in `complementary_cats_pairs/pairs.ipynb`.

## What this notebook covers

It loads the tables the model needs and turns them into training pairs. §1
reads everything; §4 builds the pair tables and §8 finishes them as
`tower_pairs_train` and `tower_pairs_test`, one row per directed
(query item, candidate item) example:

| Column | |
| --- | --- |
| `asin_query`, `query_cat_2/3/4` | the item being looked at |
| `asin_target`, `target_cat_2/3/4` | an item bought alongside it, whose category the mapping licenses |

The train/test split is temporal and was applied when `pairs.ipynb` built the
two co-purchase tables, on `date_threshold` from `ttn/constants.json`. This
notebook reads both and carries them through the same steps, producing
`tower_pairs_train` and `tower_pairs_test`.

Every statistic is **fitted on the training slice and applied to both** — the
brand fold, the eight categorical vocabularies, the price medians and the
`target_node` vocabulary. Fitting any of them twice would leave the same
category on a different integer code at evaluation time, and that code is the
embedding index. Model definition and training come next.


In [101]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# data/ lives at the repo root, one level up from this ttn/ folder
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the datasets

Everything this notebook reads, in one place. Nothing below this section opens
a file — every later cell transforms tables that are already in memory.

| Dataset | Grain | What it is |
| --- | --- | --- |
| `Home_and_Kitchen_filtered.csv` | one row per review | the interaction log — who bought what, when |
| `df_features.pkl` | one row per `asin` | extracted item attributes: `cat_*`, `brand`, `title_cleaned`, the per-field columns and their parsed measures |
| `co_purchase_pairs_train.pkl` | one row per item pair | pairs from interactions **before** `date_threshold` — the training positives |
| `co_purchase_pairs_test.pkl` | one row per item pair | pairs from interactions **at or after** it — the held-out positives |
| `complementary_categories.pkl` | one row per directed category pair | which category buys into which, filtered by support and lift — the **complementary mapping** |
| `meta_Home_and_Kitchen_filtered.csv` | one row per `asin` | the *unfiltered* catalogue; describes 150,826 items `df_features` does not |

`asin` and `reviewerID` are pinned to `str` throughout so ids with leading
zeros (e.g. `0560467893`) survive the read.

Only five of the catalogue's fifteen columns are read. `df_features` already
carries every field the catalogue has — the catalogue's value here is
**coverage**, not extra columns: it describes 28,537 reviewed asins that have
no `df_features` row, every one of them in a `cat_3` with no extraction schema.
Reading all fifteen columns of a 2.1 GB file to use four of them is waste.

All of it together peaks at about **4.5 GB** of RAM.

Both pair tables are read here and every step below runs over both. The split
is temporal and was applied when `pairs.ipynb` built them, so nothing here
re-derives it — `date_threshold` is read only to scope the statistics §5 and
§7 fit to the training period.


In [102]:
import json
from pathlib import Path

# --- 1. Interactions: one row per review ----------------------------------
df_reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    dtype={"asin": str, "reviewerID": str},
    low_memory=False,
)

# --- 2. Item features: one row per asin, the extracted attributes ---------
df_features = pd.read_pickle(DATA_DIR / "df_features.pkl")

# --- 3. Co-purchase pairs, one table per side of the cutoff ---------------
co_pairs = {
    "train": pd.read_pickle(DATA_DIR / "co_purchase_pairs_train.pkl"),
    "test": pd.read_pickle(DATA_DIR / "co_purchase_pairs_test.pkl"),
}

# The split point, read from the same file pairs.ipynb reads. Only needed to
# scope the fitted statistics in §5 and §7 to the training period.
DATE_THRESHOLD = json.loads((Path("constants.json")).read_text())["date_threshold"]
cutoff_time = pd.Timestamp(DATE_THRESHOLD).timestamp()

# --- 4. The complementary category mapping --------------------------------
comp_cat = pd.read_pickle(DATA_DIR / "complementary_categories.pkl")

# --- 5. The unfiltered catalogue: coverage for items df_features lacks ----
meta_catalogue = pd.read_csv(
    DATA_DIR / "meta_Home_and_Kitchen_filtered.csv",
    usecols=["asin", "category", "title", "brand", "price"],
    dtype={"asin": str},
    low_memory=False,
)

for name, frame in [
    ("df_reviews", df_reviews), ("df_features", df_features),
    ("co_pairs[train]", co_pairs["train"]), ("co_pairs[test]", co_pairs["test"]),
    ("comp_cat", comp_cat),
    ("meta_catalogue", meta_catalogue),
]:
    print(f"{name:<18} {str(frame.shape):>18}")

print(f"\nunique users: {df_reviews['reviewerID'].nunique():,} | "
      f"unique items: {df_reviews['asin'].nunique():,}")
print(f"items described by df_features : {df_features['asin'].nunique():,}")
print(f"items described by the catalogue: {meta_catalogue['asin'].nunique():,}")
print(f"split point (ttn/constants.json): {DATE_THRESHOLD}")
df_reviews.head(5)

df_reviews              (6898955, 11)
df_features             (1134566, 92)
co_pairs[train]         (10595885, 2)
co_pairs[test]            (942795, 2)
comp_cat                   (5845, 11)
meta_catalogue           (1300540, 5)

unique users: 777,242 | unique items: 189,172
items described by df_features : 1,134,566
items described by the catalogue: 1,285,392
split point (ttn/constants.json): 2017-12-09


,overall,verified,reviewTime,reviewerID,asin,reviewerName,summary,unixReviewTime,vote,style,image
0,5.00,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,Linda Fahner,Five Stars,1446681600,NaN,NaN,NaN
1,3.00,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,Harry Slaughter,Meh,1430956800,2,NaN,NaN
2,5.00,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,luckyg,Recommend,1390348800,NaN,{'Color:': ' Brushed Stainless'},NaN
3,1.00,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,Nickleen,Not keeping coffee hot for long enough,1383091200,NaN,{'Color:': ' Brushed Stainless'},NaN
4,1.00,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,Lacemaker427,Leaks like a waterfall when at an angle!,1379635200,NaN,{'Color:': ' Red'},NaN


## 2. Validate the feature table

Before anything is joined, check that `df_features.pkl` is what the pipeline
promised: the exact expected column set, one row per `asin`, numeric columns
actually numeric, coverage above its floors, unit columns free of new values,
and every `_cleaned` column inside its bound.

A **FAIL on `columns`** is the one to care about most — it means a feature
appeared that nothing describes, or one silently disappeared.

In [103]:
from feature_extraction_workflow.validations import run_all

report = run_all(df_features, DATA_DIR / "master_metadata.json")
display(report if len(report) else "no findings")

1,134,566 rows x 92 columns — 1 failure(s), 0 warning(s)


,check,level,subject,detail
0,columns,FAIL,also_buy,present in the table but not in the contract


## 3. Clean the item category path

`cat_4_clean` is built from `data/category_taxonomy.json` — a reviewed
whitelist of which `(cat_3, cat_4)` pairs are real categories rather than
product bullets that leaked into the path. 921 → 451 distinct values, with
0.1% of items landing in a `<cat_3>_Other` bucket.

This runs before anything else because the complementary mapping's `cat_4`
values are folded the same way. Joining §4 on the raw `cat_4` would match on
spelling rather than meaning and drop roughly one valid pair in six.


In [104]:
import json

TAXONOMY_PATH = DATA_DIR / "category_taxonomy.json"
with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

# cat_3 -> the set of cat_4 values that survived the review
valid_cat_4 = {c3: set(vals) for c2 in taxonomy for c3, vals in taxonomy[c2].items()}
print(f"taxonomy: {len(taxonomy)} cat_2 | {len(valid_cat_4)} cat_3 | "
      f"{sum(len(v) for v in valid_cat_4.values())} valid cat_4 slots")

MISSING = "Missing"
OTHER_SUFFIX = "_Other"

cat_3 = df_features["cat_3"].astype(str)
cat_4 = df_features["cat_4"].fillna(MISSING).astype(str)

# A value is kept only if it is valid *under its own parent* — the same label
# can be real in one branch and junk in another.
valid_pairs = {(c3, v) for c3, vals in valid_cat_4.items() for v in vals}
keep = pd.Series(list(zip(cat_3, cat_4)), index=df_features.index).isin(valid_pairs)

df_features["cat_4_clean"] = np.where(keep, cat_4, cat_3 + OTHER_SUFFIX)

n_before = df_features["cat_4"].nunique(dropna=False)
n_after = df_features["cat_4_clean"].nunique()
n_folded = int((~keep).sum())
print(f"\ndistinct cat_4 : {n_before:,} -> {n_after:,}")
print(f"items folded into '<cat_3>{OTHER_SUFFIX}': {n_folded:,} "
      f"({n_folded / len(df_features) * 100:.2f}% of the catalog)")
df_features[["asin", "cat_2", "cat_3", "cat_4", "cat_4_clean"]].head(5)

taxonomy: 7 cat_2 | 69 cat_3 | 521 valid cat_4 slots

distinct cat_4 : 921 -> 451
items folded into '<cat_3>_Other': 1,099 (0.10% of the catalog)


,asin,cat_2,cat_3,cat_4,cat_4_clean
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,Dinnerware
1,0002020300,Home Dcor,Candles & Holders,Candles,Candles
2,0006564224,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Glassware & Drinkware
3,0009046461,Bath,Bathroom Accessories,None,Missing
4,0234937912,Home Dcor,Home Fragrance,Incense & Incense Holders,Incense & Incense Holders


## 4. Build the pair tables

`build_tower_pairs` turns one co-purchase table into directed (query, target)
rows, and runs over both slices. Three steps:

1. **Join the categories onto both ends.** Inner join on `asin`, so a pair
   survives only if both items have features.
2. **Inner join onto the mapping, in order** — `asinA`'s three categories
   against the mapping's first three (`src_*`), `asinB`'s against the last three
   (`dst_*`). Here `asinA` is the source, so it becomes the **query**.
3. **Inner join again, reversed**, which makes `asinB` the query.

Then concatenate, both halves relabelled so the source side is always
`asin_query` / `query_cat_*`. A pair licensed in both directions appears in both
tables — `X → Y` and `Y → X` are two different examples, not duplicates.

**Nothing here is fitted.** Every step is a per-row join against a fixed
mapping, so running it on the test slice introduces no leakage. Everything that
*is* fitted — the brand threshold, the vocabularies, the price medians —
comes in §§5–8 and is learned from the training slice alone.

Categories stay plain strings at this stage. Making them `category` dtype is
itself a fitting step: the vocabulary decides which integer each value gets, and
that has to be learned once from train and applied to test, never derived twice.


In [105]:
from complementary_cats_pairs import DST_COLS, SRC_COLS

CAT_LEVELS = ["cat_2", "cat_3", "cat_4_clean"]     # matches the mapping's levels
QUERY_CATS = ["query_cat_2", "query_cat_3", "query_cat_4"]
TARGET_CATS = ["target_cat_2", "target_cat_3", "target_cat_4"]
PAIR_COLS = ["asin_query", "asin_target"] + QUERY_CATS + TARGET_CATS

item_cats = df_features[["asin"] + CAT_LEVELS]
mapping = comp_cat[SRC_COLS + DST_COLS].astype(str)


def build_tower_pairs(pairs):
    """Co-purchase pairs -> directed (query, target) rows the mapping licenses."""
    with_cats = (
        pairs
        .assign(asinA=pairs["asinA"].astype(str), asinB=pairs["asinB"].astype(str))
        .merge(item_cats.add_prefix("a_"), left_on="asinA", right_on="a_asin", how="inner")
        .merge(item_cats.add_prefix("b_"), left_on="asinB", right_on="b_asin", how="inner")
        .drop(columns=["a_asin", "b_asin"])
    )
    a_cats = [f"a_{c}" for c in CAT_LEVELS]
    b_cats = [f"b_{c}" for c in CAT_LEVELS]

    def directed(query_asin, query_cats, target_asin, target_cats):
        out = with_cats.merge(mapping, left_on=query_cats + target_cats,
                              right_on=SRC_COLS + DST_COLS, how="inner")
        return out.rename(columns=dict(
            [(query_asin, "asin_query"), (target_asin, "asin_target")]
            + list(zip(query_cats, QUERY_CATS))
            + list(zip(target_cats, TARGET_CATS))))[PAIR_COLS]

    both = pd.concat([directed("asinA", a_cats, "asinB", b_cats),
                      directed("asinB", b_cats, "asinA", a_cats)], ignore_index=True)
    return both, len(with_cats)


slices = {}
for name, pairs in co_pairs.items():
    slices[name], joined = build_tower_pairs(pairs)
    print(f"{name:<6} {len(pairs):>11,} pairs -> {joined:>11,} with features both ends "
          f"({joined / len(pairs):>5.1%}) -> {len(slices[name]):>10,} directed rows")

held_out = len(slices["test"]) / sum(len(v) for v in slices.values())
print(f"\nheld out: {held_out:.1%}")
slices["train"].head(5)

train   10,595,885 pairs ->   7,729,936 with features both ends (73.0%) ->  2,916,635 directed rows
test       942,795 pairs ->     696,070 with features both ends (73.8%) ->    270,957 directed rows

held out: 8.5%


,asin_query,asin_target,query_cat_2,query_cat_3,query_cat_4,target_cat_2,target_cat_3,target_cat_4
0,0560467893,B007EAROSK,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers
1,0560467893,B0150ZOXEI,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers
2,0681795107,B000Z4ETF8,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware
3,0681795107,B003ZYGQVK,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,Cookware,Canning
4,0681795107,B00X5ETKU4,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers


## 5. Fold rare brands — **fitted on train**

`brand_clean` folds brands carried by ten or fewer items into `other_brands`,
taking 98,532 brands down to roughly 12,700.

The threshold is counted over **items that appear in the training pairs only**.
Counting over the whole catalogue would let the test period decide which brands
are common enough to keep, which is exactly the kind of quiet leak that is hard
to find later. The resulting fold is then applied to every item, test included —
fit once on train, apply to both.


In [106]:
# Fold rare brands: keep those carried by MORE THAN 10 distinct items. A brand
# on three items would get an embedding trained by a handful of gradient
# updates; bucketing those into one `other_brands` symbol is more honest.
MIN_ITEMS = 10
OTHER = "other_brands"

# FITTED ON TRAIN. The count is taken over items appearing in the training
# pairs, so the test period has no say in which brands survive.
train_items = pd.unique(pd.concat(
    [slices["train"]["asin_query"], slices["train"]["asin_target"]], ignore_index=True))

# Count on brand_norm where the pipeline produced it, so "3d rose" and "3drose"
# are not counted separately and pushed under the threshold by a split spelling.
source = "brand_norm" if "brand_norm" in df_features.columns else "brand"
train_features = df_features[df_features["asin"].isin(train_items)]
brand_counts = train_features.groupby(source)["asin"].nunique()
kept_brands = brand_counts[brand_counts > MIN_ITEMS].index

# APPLIED TO ALL ITEMS, so test rows are folded by the training rule.
df_features["brand_clean"] = df_features[source].where(
    df_features[source].isin(kept_brands) | df_features[source].isna(), OTHER)

print(f"counted on   : {source}, over {len(train_items):,} training items")
print(f"brands kept  : {len(kept_brands):,} of "
      f"{train_features[source].nunique():,} seen in training")
print(f"all items    : {(df_features['brand_clean'] == OTHER).mean():.1%} in {OTHER}, "
      f"{df_features[source].isna().mean():.1%} missing (left as NaN for §7)")
print()
print(df_features["brand_clean"].value_counts().head(10))

counted on   : brand, over 146,984 training items
brands kept  : 2,322 of 25,726 seen in training
all items    : 50.9% in other_brands, 5.7% missing (left as NaN for §7)

brand_clean
other_brands     577792
3dRose             8569
CafePress          7489
Disney             6249
Unknown            5594
Hallmark           4720
Generic            4500
Department 56      3416
Safavieh           3227
Kurt Adler         3124
Name: count, dtype: int64


## 6. Attach the item attributes, and fit the vocabularies

Each side gains seven columns — the item's own content, which is what the
product encoder reads. Names are lowercase throughout:

| Column | Source | Coverage on paired items |
| --- | --- | --- |
| `*_title_cleaned` | `title_cleaned` | 100% |
| `*_price` | `price`, parsed to a number | 75% |
| `*_brand_clean` | `brand_clean` from §5 | 99% |
| `*_product_type` | `Product_Type` | 76.5% |
| `*_material` | `Material` | 72.5% |
| `*_features` | `Features` | 61.7% |
| `*_color` | `Color` | 54.2% |

**`price` needs parsing, not just carrying.** It is stored as a string
(`'$37.00'`), and 7,032 of its non-null values are not prices at all but scraped
CSS. Stripping the currency symbol and coercing handles both: real prices become
floats, the junk becomes `NaN`.

### The vocabularies are the fitted part

Every categorical — the three category levels plus the five string attributes —
gets **one `CategoricalDtype` fitted on the training slice** and applied to
both. Two properties depend on this and both are silent failures if got wrong:

- **Across towers.** A value must index the same embedding row whether it
  arrives as a query or a target. Casting each column independently gives them
  different category lists, which is what made `query_cat_3 == target_cat_3`
  raise earlier — and would have had `Clocks` at code 1 in one tower and code 0
  in the other.
- **Across slices.** The same applies between train and test. Fitting the
  vocabulary twice would leave a category on a different code at evaluation
  time, so the model would look up the wrong vector for a value it knows
  perfectly well.

A test value outside the fitted vocabulary becomes `NaN` on cast and is filled
in §7. That is rare here — 236 rows in 541,914, about 0.04% — and §7 prints the
count so it cannot start growing unnoticed.


In [107]:
# --- Item attributes, one row per asin ------------------------------------
# Source column in df_features -> the name it takes in the pair tables.
ITEM_ATTRS = {
    "title_cleaned": "title_cleaned",
    "price": "price",
    "brand_clean": "brand_clean",
    "Color": "color",
    "Features": "features",
    "Material": "material",
    "Product_Type": "product_type",
}
CATEGORICAL_ATTRS = ["cat_2", "cat_3", "cat_4", "brand_clean",
                     "color", "features", "material", "product_type"]
SIDE_COLS = ["asin_{s}", "{s}_cat_2", "{s}_cat_3", "{s}_cat_4",
             "{s}_title_cleaned", "{s}_price", "{s}_brand_clean",
             "{s}_color", "{s}_features", "{s}_material", "{s}_product_type"]

item_attrs = (df_features[["asin"] + list(ITEM_ATTRS)].rename(columns=ITEM_ATTRS))
item_attrs["price"] = pd.to_numeric(
    item_attrs["price"].astype(str).str.replace(r"[$,]", "", regex=True),
    errors="coerce")
attrs_by_asin = item_attrs.set_index("asin")
print(f"price parsed : {item_attrs['price'].notna().sum():,} of {len(item_attrs):,} "
      f"items ({item_attrs['price'].notna().mean():.1%})")


def attach_attributes(frame):
    """Item content for both sides. `.map`, not `merge`, so re-running is safe."""
    for side in ("query", "target"):
        asin = frame[f"asin_{side}"]
        for attr in ITEM_ATTRS.values():
            frame[f"{side}_{attr}"] = asin.map(attrs_by_asin[attr])
    return frame[[c.format(s=s) for s in ("query", "target") for c in SIDE_COLS]]


for name in slices:
    slices[name] = attach_attributes(slices[name])

# --- Fit the vocabularies on TRAIN, apply to both -------------------------
# `title_cleaned` is excluded: it is free text for the encoder, not a symbol to
# look up, and 145k levels would only cost memory.
vocabularies = {}
for attr in CATEGORICAL_ATTRS:
    values = set()
    for side in ("query", "target"):
        values |= set(slices["train"][f"{side}_{attr}"].dropna().astype(str))
    vocabularies[attr] = pd.CategoricalDtype(sorted(values))

print(f"\nvocabularies fitted on {len(slices['train']):,} training rows:")
for attr, dtype in vocabularies.items():
    print(f"  {attr:<14} {len(dtype.categories):>8,} levels")
slices["train"].head(5)

price parsed : 515,980 of 1,134,566 items (45.5%)

vocabularies fitted on 2,916,635 training rows:
  cat_2                 7 levels
  cat_3                69 levels
  cat_4               395 levels
  brand_clean       2,323 levels
  color                63 levels
  features            542 levels
  material            211 levels
  product_type      1,106 levels


,asin_query,query_cat_2,query_cat_3,query_cat_4,query_title_cleaned,query_price,query_brand_clean,query_color,query_features,query_material,query_product_type,asin_target,target_cat_2,target_cat_3,target_cat_4,target_title_cleaned,target_price,target_brand_clean,target_color,target_features,target_material,target_product_type
0,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,NaN,WELLAND,black,floating shelf,None,corner shelf,B007EAROSK,Bath,Bathroom Accessories,Holders & Dispensers,simplehuman precision lever square push soap p...,15.99,simplehuman,None,None,plastic,soap pump
1,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,NaN,WELLAND,black,floating shelf,None,corner shelf,B0150ZOXEI,Bath,Bathroom Accessories,Holders & Dispensers,double toothbrush holder angle simple sus304 s...,22.99,other_brands,None,wall mounted,stainless steel,toothbrush holder
2,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,None,None,stainless,mug,B000Z4ETF8,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,tervis 1001833 clear colorful insulated tumble...,24.00,other_brands,clear,insulated,melamine,tumbler
3,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,None,None,stainless,mug,B003ZYGQVK,Kitchen & Dining,Cookware,Canning,tervis travel lid 24 oz orange,7.37,Tervis,orange,dishwasher safe,None,lid
4,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,None,None,stainless,mug,B00X5ETKU4,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers,mr coffee 12-cup programmable coffee maker the...,NaN,Mr. Coffee,None,carafe,chrome,coffee maker


## 7. Fill the missing values — **fitted on train**

### Prices — the category median

About a quarter of rows have no price. Each takes the median price of its own
category, from a table fitted on the training period alone.

**The population is items reviewed before the cutoff**, not the whole
catalogue. Two consequences, and the first is why:

- *Temporally clean.* Nothing that happens at or after `date_threshold` can
  influence a training feature. On this data the effect is small — only 277
  items are test-period-only, moving 2 of 697 medians by more than $1 — but it
  is the difference between a guarantee and a measurement that has to be
  re-checked whenever the split moves.
- *Smaller and noisier.* Restricting from ~1.29M catalogue items to the ~189k
  that were actually reviewed cuts the sample behind each median roughly in
  half and leaves some paths unpriced. The **fallback ladder**
  `cat_4 → cat_3 → cat_2 → global` closes those.

`*_price_imputed` records which values were inferred, because a quarter of the
column is being filled and whether a price is real is itself a signal.

### Categoricals — the fitted vocabulary, then an explicit `Missing`

Applying §6's train-fitted `CategoricalDtype` does two jobs at once. Values in
the vocabulary keep a stable code across both towers and both slices. Values
outside it — only possible in test — become `NaN` on cast and are then filled
with `Missing`, alongside the genuinely absent ones.

Folding unseen values into `Missing` rather than a separate `Unknown` is
deliberate. `Unknown` would carry ~236 training examples, so its embedding would
never leave its initialisation and whatever the model did with it would be
arbitrary. `Missing` is well trained and means "no usable value for this
attribute", which is a fair description of the situation. The cell prints the
unseen count so that if it ever grows past a fraction of a percent, the decision
can be revisited.

⚠️ Every statistic in this section is fitted on `slices["train"]` and applied to
both. Nothing is recomputed on the test slice.


In [108]:
from complementary_cats_pairs import fold_cat_4, parse_category_levels

# --- Category price medians: FITTED ON TRAIN ------------------------------
# Population: items reviewed strictly before the cutoff. The category path is
# parsed and folded exactly as §3 folds df_features, so the medians key onto
# query_cat_2/3/4 and target_cat_2/3/4 directly.
train_reviewed = set(df_reviews.loc[df_reviews["unixReviewTime"] < cutoff_time, "asin"])

cat_levels = parse_category_levels(meta_catalogue["category"], n_levels=4)
catalogue_prices = pd.DataFrame({
    "asin": meta_catalogue["asin"],
    "cat_2": cat_levels["cat_2"].astype(str),
    "cat_3": cat_levels["cat_3"].astype(str),
    "cat_4": fold_cat_4(cat_levels["cat_3"], cat_levels["cat_4"], valid_pairs),
    "price": pd.to_numeric(
        meta_catalogue["price"].astype(str).str.replace(r"[$,]", "", regex=True),
        errors="coerce"),
})
fit_prices = catalogue_prices[catalogue_prices["asin"].isin(train_reviewed)]

median_4 = fit_prices.groupby(["cat_2", "cat_3", "cat_4"], observed=True)["price"].median()
median_3 = fit_prices.groupby(["cat_2", "cat_3"], observed=True)["price"].median()
median_2 = fit_prices.groupby(["cat_2"], observed=True)["price"].median()
median_all = fit_prices["price"].median()

print(f"median population : {len(fit_prices):,} items reviewed before "
      f"{DATE_THRESHOLD} ({fit_prices['price'].notna().sum():,} priced)")
print(f"median tables     : {median_4.notna().sum():,} cat_4 paths | "
      f"{median_3.notna().sum()} cat_3 | {median_2.notna().sum()} cat_2 | "
      f"global ${median_all:,.2f}")


def fill_prices(frame):
    """Fill each missing price from its category median, falling back up."""
    for side in ("query", "target"):
        raw = frame[f"asin_{side}"].map(attrs_by_asin["price"])
        c2 = frame[f"{side}_cat_2"].astype(str)
        c3 = frame[f"{side}_cat_3"].astype(str)
        c4 = frame[f"{side}_cat_4"].astype(str)
        at_4 = pd.Series(median_4.reindex(pd.MultiIndex.from_arrays([c2, c3, c4])).to_numpy(),
                         index=frame.index)
        at_3 = pd.Series(median_3.reindex(pd.MultiIndex.from_arrays([c2, c3])).to_numpy(),
                         index=frame.index)
        at_2 = pd.Series(median_2.reindex(pd.Index(c2)).to_numpy(), index=frame.index)
        missing = raw.isna()
        frame[f"{side}_price"] = raw.fillna(
            at_4.fillna(at_3).fillna(at_2).fillna(median_all))
        frame[f"{side}_price_imputed"] = missing
        yield side, missing, at_4, at_3, at_2


def apply_vocabularies(frame):
    """Cast to the train-fitted dtypes; unseen values fall out as NaN."""
    unseen = {}
    for attr, dtype in vocabularies.items():
        for side in ("query", "target"):
            col = f"{side}_{attr}"
            values = frame[col].astype(object)
            cast = pd.Categorical(values, dtype=dtype)
            unseen[attr] = unseen.get(attr, 0) + int(
                (values.notna() & pd.isna(cast)).sum())
            frame[col] = cast
    return unseen


for name, frame in slices.items():
    print(f"\n--- {name} ---")
    for side, missing, at_4, at_3, at_2 in fill_prices(frame):
        rungs = (int((missing & at_4.notna()).sum()),
                 int((missing & at_4.isna() & at_3.notna()).sum()),
                 int((missing & at_4.isna() & at_3.isna() & at_2.notna()).sum()),
                 int((missing & at_4.isna() & at_3.isna() & at_2.isna()).sum()))
        print(f"  {side:<7} {missing.sum():>9,} missing ({missing.mean():>5.1%}) -> "
              f"cat_4 {rungs[0]:,} | cat_3 {rungs[1]:,} | cat_2 {rungs[2]:,} | "
              f"global {rungs[3]:,} | still NaN "
              f"{frame[f'{side}_price'].isna().sum():,}")
    unseen = apply_vocabularies(frame)
    total_unseen = sum(unseen.values())
    print(f"  values outside the training vocabulary: {total_unseen:,} "
          f"({total_unseen / (len(frame) * 2 * len(vocabularies)):.4%} of slots)"
          + (f"  {ute}" if (ute := {k: v for k, v in unseen.items() if v}) else ""))

median population : 193,265 items reviewed before 2017-12-09 (120,916 priced)
median tables     : 584 cat_4 paths | 159 cat_3 | 13 cat_2 | global $16.45

--- train ---
  query     726,061 missing (24.9%) -> cat_4 726,061 | cat_3 0 | cat_2 0 | global 0 | still NaN 0
  target    721,768 missing (24.7%) -> cat_4 721,768 | cat_3 0 | cat_2 0 | global 0 | still NaN 0
  values outside the training vocabulary: 0 (0.0000% of slots)

--- test ---
  query      38,727 missing (14.3%) -> cat_4 38,727 | cat_3 0 | cat_2 0 | global 0 | still NaN 0
  target     38,301 missing (14.1%) -> cat_4 38,301 | cat_3 0 | cat_2 0 | global 0 | still NaN 0
  values outside the training vocabulary: 6 (0.0001% of slots)  {'features': 4, 'product_type': 2}


In [109]:
MISSING_LABEL = "Missing"


def fill_missing_categories(frame, attrs, label=MISSING_LABEL):
    """Give every missing categorical value an explicit `label` level.

    Covers both the genuinely absent and anything that fell outside the
    train-fitted vocabulary in the cell above -- after the cast both are NaN,
    and both mean the same thing to the encoder: no usable value here.

    A pandas Categorical refuses a value that is not one of its categories, so
    the level is added before it is assigned. Idempotent.
    """
    for attr in attrs:
        for side in ("query", "target"):
            col = f"{side}_{attr}"
            series = frame[col]
            if label not in series.cat.categories:
                series = series.cat.add_categories([label])
            frame[col] = series.fillna(label)
    return frame


for name, frame in slices.items():
    before = {c: frame[c].isna().sum()
              for c in (f"{s}_{a}" for s in ("query", "target") for a in CATEGORICAL_ATTRS)}
    slices[name] = fill_missing_categories(frame, CATEGORICAL_ATTRS)
    filled = sum(before.values())
    print(f"{name:<6} filled {filled:>9,} categorical values with {MISSING_LABEL!r} "
          f"({filled / (len(frame) * 2 * len(CATEGORICAL_ATTRS)):.2%} of slots)")

print("\nvocabularies identical across towers and slices:")
for attr in CATEGORICAL_ATTRS:
    levels = [list(slices[n][f"{s}_{attr}"].cat.categories)
              for n in slices for s in ("query", "target")]
    print(f"  {attr:<14} {all(l == levels[0] for l in levels)}  "
          f"({len(levels[0]):,} levels, {MISSING_LABEL!r} present: "
          f"{MISSING_LABEL in levels[0]})")

train  filled 7,096,407 categorical values with 'Missing' (15.21% of slots)
test   filled   589,876 categorical values with 'Missing' (13.61% of slots)

vocabularies identical across towers and slices:
  cat_2          True  (8 levels, 'Missing' present: True)
  cat_3          True  (70 levels, 'Missing' present: True)
  cat_4          True  (395 levels, 'Missing' present: True)
  brand_clean    True  (2,324 levels, 'Missing' present: True)
  color          True  (64 levels, 'Missing' present: True)
  features       True  (543 levels, 'Missing' present: True)
  material       True  (212 levels, 'Missing' present: True)
  product_type   True  (1,107 levels, 'Missing' present: True)


## 8. `target_node` — the target category as one symbol

The three target levels collapsed into a single value, e.g.
`Bath > Bathroom Accessories > Holders & Dispensers`.

This is the **target category** the paper's query tower consumes: in
Complementary-TT the query product embedding is concatenated with a target
category embedding, and that embedding needs one symbol per category to look
up, not three separate levels. Keeping the full path rather than `cat_4` alone
matters because a leaf label is not unique on its own — `Missing` and
`<cat_3>_Other` recur under many parents, so `cat_4` by itself would collapse
genuinely different categories onto one embedding row.

Built **after** §7, deliberately: every level is complete by then, so no node
carries a `NaN` fragment or silently becomes the string `"nan"`.

Its vocabulary is **fitted on train** like every other, and a test node outside
it falls back to `Missing` — the same rule §7 applies to the individual levels.
Because §4 only keeps pairs whose category path is in the mapping, that should
never fire; the cell asserts it rather than assuming.

Finally the two slices are named `tower_pairs_train` and `tower_pairs_test`.
There is no bare `tower_pairs` — with two slices in play the unqualified name
would be ambiguous about which one it meant.


In [ ]:
# One symbol per target category path, for the query tower's target-category
# embedding. `" > "` is cosmetic -- it only has to be readable and not appear
# inside a category name.
NODE_SEP = " > "
TARGET_LEVELS = ["target_cat_2", "target_cat_3", "target_cat_4"]


def target_node(frame):
    return (frame["target_cat_2"].astype(str) + NODE_SEP
            + frame["target_cat_3"].astype(str) + NODE_SEP
            + frame["target_cat_4"].astype(str))


# FITTED ON TRAIN, applied to both.
node_vocabulary = pd.CategoricalDtype(sorted(set(target_node(slices["train"]))))
print(f"target_node vocabulary: {len(node_vocabulary.categories):,} paths, "
      f"fitted on {len(slices['train']):,} training rows")

for name, frame in slices.items():
    nodes = target_node(frame)
    cast = pd.Categorical(nodes, dtype=node_vocabulary)
    unseen = int(pd.isna(cast).sum())
    if unseen:                     # never expected: §4 filters on the mapping
        cast = cast.add_categories([MISSING_LABEL]).fillna(MISSING_LABEL)
    frame["target_node"] = cast
    print(f"  {name:<6} {nodes.nunique():>4,} distinct | outside the training "
          f"vocabulary: {unseen:,}")

# --- The two finished tables ----------------------------------------------
tower_pairs_train = slices["train"]
tower_pairs_test = slices["test"]

for name, frame in (("train", tower_pairs_train), ("test", tower_pairs_test)):
    print(f"\n{name:<6} {str(frame.shape):>16} | "
          f"{frame.memory_usage(deep=True).sum() / 1e6:>6,.0f} MB | "
          f"price complete: "
          f"{frame['query_price'].notna().all() and frame['target_price'].notna().all()}")
print(f"\ncolumns ({tower_pairs_train.shape[1]}): {list(tower_pairs_train.columns)}")
tower_pairs_train.head(5)

## 9. Export the model-ready arrays

Turns the two tables into what a training loop wants: **pairs hold integer
indices only, and every feature lives once per item.** A pair table repeating a
384-dim title vector on both sides would be 2.9M × 768 floats — 9 GB — to say
the same thing 160k rows of `items.npz` say.

Written to `data/tower/`:

| File | Contents |
| --- | --- |
| `items.npz` | `title_emb` (n × 384), `cat_ids` (n × 8), `numeric` (n × 3) |
| `pairs_train.parquet`, `pairs_test.parquet` | `query_idx`, `target_idx`, `target_node_id`, `weight` |
| `vocabs.json` | every categorical vocabulary, so training and serving agree |
| `node_of_item.npy` | each item's own target-category id |

**Vocabulary id 0 is reserved** as the padding/unseen index, so every real value
starts at 1. §7 already folded unseen values into `Missing`, so 0 should never
be produced here — the cell asserts that rather than trusting it.

### Three fixes against the draft script

- **`price` was read from `price_imputed`.** That column is the boolean flag
  §7 sets, not a price, so `log1p` was being taken of 0/1 and the actual price
  discarded. The numeric block now reads `price` and uses `price_imputed` as the
  missingness indicator — which also restores the signal, since `price.isna()`
  is uniformly `False` after §7 and would have made that feature a column of
  zeros.
- **`node_of_item` was scattered from the training pairs**, leaving every item
  that never appears as a *target* at id 0. A node is a function of the item's
  own category path, so it is derived directly from `items` and is complete.
- **The price decile was divided by 10** while `duplicates="drop"` can return
  fewer than ten bins, so the feature was not on `[0, 1]`. It is normalised by
  the bin count actually produced.

Titles reuse the 384-dim SBERT vectors already in
`df_features_with_embeddings.pkl` rather than re-encoding — same model, and it
is the vector `embedding_analysis/` produced. Set `ENCODE_TITLES = True` to
compute them instead (~2.5 min on CPU); the precomputed path costs a transient
4.8 GB while the frame is open.


In [ ]:
from pathlib import Path

OUT_DIR = DATA_DIR / "tower"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ENCODE_TITLES = False          # True -> run SBERT instead of reusing the pickle

CAT_ORDER = ["cat_2", "cat_3", "cat_4", "brand", "color", "material",
             "product_type", "features"]
# column in the pair tables (without the query_/target_ prefix) -> canonical name
COLMAP = {"cat_2": "cat_2", "cat_3": "cat_3", "cat_4": "cat_4",
          "brand_clean": "brand", "color": "color", "material": "material",
          "product_type": "product_type", "features": "features"}


def _side(df, prefix):
    cols = {f"{prefix}_{src}": dst for src, dst in COLMAP.items()}
    cols[f"asin_{prefix}"] = "asin"
    cols[f"{prefix}_title_cleaned"] = "title"
    cols[f"{prefix}_price"] = "price"
    cols[f"{prefix}_price_imputed"] = "price_imputed"
    return df[list(cols)].rename(columns=cols)


def build_items(train, test):
    """One row per asin, from every side of both slices."""
    frames = [_side(d, p) for d in (train, test) for p in ("query", "target")]
    items = (pd.concat(frames, ignore_index=True)
               .drop_duplicates("asin").reset_index(drop=True))
    items["idx"] = np.arange(len(items))
    return items


def fit_vocabs(train):
    """FIT ON TRAIN ONLY. Id 0 is reserved for padding / unseen."""
    vocabs = {}
    for src, dst in COLMAP.items():
        values = pd.unique(pd.concat(
            [train[f"query_{src}"].astype(str), train[f"target_{src}"].astype(str)],
            ignore_index=True))
        vocabs[dst] = {v: i + 1 for i, v in enumerate(sorted(values))}
    nodes = sorted(train["target_node"].astype(str).unique())
    vocabs["target_node"] = {v: i + 1 for i, v in enumerate(nodes)}
    return vocabs


def encode_items(items, vocabs, title_emb):
    cat_ids = np.zeros((len(items), len(CAT_ORDER)), dtype=np.int64)
    for j, name in enumerate(CAT_ORDER):
        cat_ids[:, j] = items[name].astype(str).map(vocabs[name]).fillna(0).to_numpy()

    # FIX: the price, not the imputation flag. `price_imputed` is the indicator.
    price = items["price"].astype(float)
    imputed = items["price_imputed"].astype("float32")
    price = price.fillna(price.median())          # defensive; §7 leaves none

    # FIX: normalise by the bins qcut actually produced, not a hardcoded 10.
    decile = pd.qcut(price, 10, labels=False, duplicates="drop")
    decile = decile.fillna(decile.median())
    numeric = np.stack([
        np.log1p(price.to_numpy()).astype("float32"),
        (decile.to_numpy() / max(decile.max(), 1)).astype("float32"),
        imputed.to_numpy(),
    ], axis=1).astype("float32")
    return {"title_emb": title_emb.astype("float32"),
            "cat_ids": cat_ids, "numeric": numeric}


def slim_pairs(df, items, vocabs):
    idx = dict(zip(items["asin"], items["idx"]))
    out = pd.DataFrame({
        "query_idx": df["asin_query"].map(idx),
        "target_idx": df["asin_target"].map(idx),
        "target_node_id": df["target_node"].astype(str)
                            .map(vocabs["target_node"]).fillna(0).astype(int),
        "weight": df.get("weight", pd.Series(1.0, index=df.index)).astype("float32"),
    }).dropna()
    return out.astype({"query_idx": int, "target_idx": int})


def item_title_embeddings(items):
    if ENCODE_TITLES:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("all-MiniLM-L6-v2")
        return np.asarray(model.encode(items["title"].astype(str).fillna("").tolist(),
                                       batch_size=256, show_progress_bar=True))
    # Reuse the vectors embedding_analysis/ already produced (384-d, unit-norm).
    # Narrowed to our asins before anything is stacked: the pickle holds 1.13M
    # vectors and we need ~160k, so a lookup over the whole frame is the
    # difference between seconds and minutes.
    wanted = set(items["asin"])
    frame = pd.read_pickle(DATA_DIR / "df_features_with_embeddings.pkl")[
        ["asin", "title_embedding"]]
    frame = frame[frame["asin"].isin(wanted)]
    vectors = frame.set_index("asin")["title_embedding"].reindex(items["asin"])
    del frame

    found = vectors.notna().to_numpy()
    dim = len(vectors[found].iloc[0])
    out = np.zeros((len(items), dim), dtype="float32")
    out[found] = np.stack(vectors[found].to_numpy()).astype("float32")
    print(f"title vectors: {int(found.sum()):,} matched, "
          f"{int((~found).sum()):,} missing (left as zeros)")
    return out


# --------------------------------------------------------------------------
items = build_items(tower_pairs_train, tower_pairs_test)
vocabs = fit_vocabs(tower_pairs_train)               # TRAIN ONLY
print(f"items: {len(items):,} unique asins across both slices")

title_emb = item_title_embeddings(items)
arrays = encode_items(items, vocabs, title_emb)

pairs_train = slim_pairs(tower_pairs_train, items, vocabs)
pairs_test = slim_pairs(tower_pairs_test, items, vocabs)

# FIX: a node is a function of the item's own path, so derive it for every
# item. Scattering from the training pairs left query-only items at 0.
item_node = (items["cat_2"].astype(str) + NODE_SEP + items["cat_3"].astype(str)
             + NODE_SEP + items["cat_4"].astype(str))
node_of_item = item_node.map(vocabs["target_node"]).fillna(0).astype(np.int64).to_numpy()

np.savez_compressed(OUT_DIR / "items.npz", **arrays)
json.dump(vocabs, open(OUT_DIR / "vocabs.json", "w"))
pairs_train.to_parquet(OUT_DIR / "pairs_train.parquet")
pairs_test.to_parquet(OUT_DIR / "pairs_test.parquet")
np.save(OUT_DIR / "node_of_item.npy", node_of_item)

# §7 folded every unseen value into Missing, so nothing should land on the
# reserved id. Assert rather than assume.
assert (arrays["cat_ids"] > 0).all(), "a categorical fell on the reserved id 0"
assert (pairs_train["target_node_id"] > 0).all(), "a train node fell on id 0"
unseen_nodes = int((pairs_test["target_node_id"] == 0).sum())

print(f"\nitems {len(items):,} | train {len(pairs_train):,} | test {len(pairs_test):,}")
print(f"title_emb {arrays['title_emb'].shape} | cat_ids {arrays['cat_ids'].shape} "
      f"| numeric {arrays['numeric'].shape}")
print(f"vocab sizes: {({k: len(v) for k, v in vocabs.items()})}")
print(f"items with no node in the training vocabulary: {(node_of_item == 0).sum():,}")
print(f"test pairs whose target node is unseen       : {unseen_nodes:,}")
print(f"\nwritten to {OUT_DIR.relative_to(ROOT)}/")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:<24} {f.stat().st_size / 1e6:>8,.1f} MB")

In [111]:
print(f'Shape of the table: {tower_pairs_train.shape}')
print(f'Variables of the table: {tower_pairs_train.columns}')

Shape of the table: (2916635, 25)
Variables of the table: Index(['asin_query', 'query_cat_2', 'query_cat_3', 'query_cat_4',
       'query_title_cleaned', 'query_price', 'query_brand_clean',
       'query_color', 'query_features', 'query_material', 'query_product_type',
       'asin_target', 'target_cat_2', 'target_cat_3', 'target_cat_4',
       'target_title_cleaned', 'target_price', 'target_brand_clean',
       'target_color', 'target_features', 'target_material',
       'target_product_type', 'query_price_imputed', 'target_price_imputed',
       'target_node'],
      dtype='object')


In [115]:
tower_pairs_test.shape

(270957, 25)